In [8]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from io import BytesIO
import scipy.ndimage as ndimage
import tensorflow as tf
from tensorflow.keras.applications import VGG16
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing import image
from sklearn.model_selection import StratifiedGroupKFold, train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc, accuracy_score
from sklearn.preprocessing import StandardScaler, label_binarize
import xgboost as xgb
from tqdm import tqdm
import time
import warnings
warnings.filterwarnings('ignore')

# 1. FIX: ENSURE REPRODUCIBILITY FOR PEER REVIEW
np.random.seed(42)

# ====================== 1. LOAD DATA ======================
print("Loading datasets...")
try:
    df_1m = pd.read_csv('sapimouse_ABS_dx_dy_1min.csv', header=None)
    df_3m = pd.read_csv('sapimouse_ABS_dx_dy_3min.csv', header=None)
    df = pd.concat([df_1m, df_3m], ignore_index=True)
    print(f"✅ Successfully merged datasets. Combined data shape: {df.shape}")
except Exception as e:
    df = pd.read_csv('sapimouse_ABS_dx_dy_1min.csv', header=None)
    print(f"✅ Data shape: {df.shape}")

# ====================== 2. TRUE 2D HELPER FUNCTIONS ======================
def trajectory_to_image(data, size=224):
    dx = data[:128]
    dy = data[128:]
    x = np.cumsum(dx)
    y = np.cumsum(dy)
    
    fig, ax = plt.subplots(figsize=(4, 4), dpi=100)
    ax.plot(x, y, color='blue', linewidth=2) 
    ax.set_aspect('equal', adjustable='datalim') 
    ax.axis('off')
    
    buf = BytesIO()
    fig.savefig(buf, format='png', bbox_inches='tight', pad_inches=0)
    buf.seek(0)
    img = image.load_img(buf, target_size=(size, size))
    plt.close(fig)
    return image.img_to_array(img)

def extract_handcrafted(data):
    dx = data[:128]
    dy = data[128:]
    x = np.cumsum(dx)
    y = np.cumsum(dy)
    
    speed = np.sqrt(dx**2 + dy**2) 
    if len(speed) == 0: speed = np.array([0])
        
    angles = np.arctan2(dy, dx)
    direction_changes = np.abs(np.diff(angles))
    
    acceleration = np.abs(np.diff(speed)) 
    
    total_dist = np.sum(speed)
    displacement = np.sqrt((x[-1]-x[0])**2 + (y[-1]-y[0])**2)
    straightness = displacement / (total_dist + 1e-9)
    
    pause_count = np.sum(speed < 0.5)
    speed_peaks = len(np.where(np.diff(np.sign(np.diff(speed))))[0])
    
    x_range = np.max(x) - np.min(x)
    y_range = np.max(y) - np.min(y)
    aspect_ratio = x_range / (y_range + 1e-9)
    
    return [
        np.mean(speed), np.std(speed), np.max(speed),
        np.mean(acceleration) if len(acceleration)>0 else 0, np.std(acceleration) if len(acceleration)>0 else 0,
        np.mean(direction_changes) if len(direction_changes)>0 else 0, 
        np.std(direction_changes) if len(direction_changes)>0 else 0,
        straightness, total_dist, pause_count, speed_peaks, aspect_ratio
    ]

# ====================== 3. VGG16 SETUP & 2D DATASET CREATION ======================
print("Loading VGG16...")
base_model = VGG16(weights='imagenet', include_top=False, pooling='avg')
feature_extractor = Model(inputs=base_model.input, outputs=base_model.output)

def create_5class_dataset(df_source):
    n_samples = len(df_source)
    # FIX: Renamed 'y' to 'labels' to prevent variable shadowing crashes
    X_fused, labels, groups = [], [], []
    
    for i in tqdm(range(n_samples), desc="Generating 2D Synthetic Adversaries"):
        row = df_source.iloc[i]
        data = row.iloc[:-1].values.astype(float) 
        user_id = int(row.iloc[-1])
        
        L = 128
        dx = data[:128]
        dy = data[128:]
        x = np.cumsum(dx)
        y = np.cumsum(dy)

        # 0. HUMAN BASELINE
        img_h = trajectory_to_image(data)
        cnn_h = feature_extractor.predict(np.expand_dims(img_h/255.0, axis=0), verbose=0)[0]
        hand_h = extract_handcrafted(data)
        X_fused.append(np.concatenate([cnn_h, hand_h]))
        labels.append(0)

        # 1. SMOOTH SCRIPTED BOT (L2) - 2D
        bot_x = ndimage.gaussian_filter1d(x, sigma=8)
        bot_y = ndimage.gaussian_filter1d(y, sigma=8)
        dx_s = np.diff(bot_x, prepend=bot_x[0]) + np.random.normal(0, 0.5, L)
        dy_s = np.diff(bot_y, prepend=bot_y[0]) + np.random.normal(0, 0.5, L)
        
        # FIX: STRUCTURAL LEAKAGE (Apply np.abs to match human dataset format)
        bot_smooth = np.concatenate([np.abs(dx_s), np.abs(dy_s)])
        
        img_s = trajectory_to_image(bot_smooth)
        cnn_s = feature_extractor.predict(np.expand_dims(img_s/255.0, axis=0), verbose=0)[0]
        hand_s = extract_handcrafted(bot_smooth)
        X_fused.append(np.concatenate([cnn_s, hand_s]))
        labels.append(1)

        # 2. WAYPOINT BOT (L2) - 2D Spatial Jumps
        dx_w, dy_w = np.zeros(L), np.zeros(L)
        num_wp = 4
        seg_len = L // num_wp
        pause_len = max(1, seg_len // 4)
        for w in range(num_wp):
            start = w * seg_len
            end = start + seg_len if w < num_wp-1 else L
            move_end = end - pause_len
            active = move_end - start
            if active > 0:
                target_dx = (x[end-1] - x[start]) / active
                target_dy = (y[end-1] - y[start]) / active
                dx_w[start:move_end] = target_dx
                dy_w[start:move_end] = target_dy
        dx_w += np.random.normal(0, 0.2, L)
        dy_w += np.random.normal(0, 0.2, L)
        
        bot_waypoint = np.concatenate([np.abs(dx_w), np.abs(dy_w)]) 
        
        img_w = trajectory_to_image(bot_waypoint)
        cnn_w = feature_extractor.predict(np.expand_dims(img_w/255.0, axis=0), verbose=0)[0]
        hand_w = extract_handcrafted(bot_waypoint)
        X_fused.append(np.concatenate([cnn_w, hand_w]))
        labels.append(2)

        # 3. KINEMATIC BOT (L3) - 2D Physics
        dx_k, dy_k = np.zeros(L), np.zeros(L)
        vx, vy = 0.0, 0.0
        cx, cy = x[0], y[0]
        for t in range(L):
            tx = x[t] if t < L else x[-1]
            ty = y[t] if t < L else y[-1]
            vx = 0.6 * vx + 0.4 * (tx - cx) + np.random.normal(0, 2.0)
            vy = 0.6 * vy + 0.4 * (ty - cy) + np.random.normal(0, 2.0)
            cx += vx * 0.1
            cy += vy * 0.1  # THE ONE-CHARACTER PHYSICS FIX
            dx_k[t] = vx
            dy_k[t] = vy
        dx_k += np.random.normal(0, 1.0, L)
        dy_k += np.random.normal(0, 1.0, L)
        
        # FIX: STRUCTURAL LEAKAGE
        bot_kin = np.concatenate([np.abs(dx_k), np.abs(dy_k)])
        
        img_kin = trajectory_to_image(bot_kin)
        cnn_kin = feature_extractor.predict(np.expand_dims(img_kin/255.0, axis=0), verbose=0)[0]
        hand_kin = extract_handcrafted(bot_kin)
        X_fused.append(np.concatenate([cnn_kin, hand_kin]))
        labels.append(3)

        # 4. SIGMA-LOGNORMAL BOT (L4) - 2D Vector Bursts
        dx_n, dy_n = np.zeros(L), np.zeros(L)
        num_strokes = max(8, L // 15)
        for s in range(num_strokes):
            center = np.random.randint(0, L)
            dur = np.random.randint(L//20, L//5)
            t_arr = np.arange(dur)
            speed_burst = np.exp(-((t_arr - dur/2)**2) / (2*(dur/6)**2))
            angle = np.random.uniform(0, 2*np.pi) 
            start = max(0, center - dur//2)
            end = min(L, start + dur)
            chunk = end - start
            dx_n[start:end] += speed_burst[:chunk] * np.cos(angle)
            dy_n[start:end] += speed_burst[:chunk] * np.sin(angle)

        orig_dist = np.sum(np.sqrt(dx**2 + dy**2))
        new_dist = np.sum(np.sqrt(dx_n**2 + dy_n**2))
        if new_dist > 0:
            dx_n = dx_n * (orig_dist / new_dist)
            dy_n = dy_n * (orig_dist / new_dist)
            
        dx_n += np.random.normal(0, 0.8, L)
        dy_n += np.random.normal(0, 0.8, L)
        
        # FIX: STRUCTURAL LEAKAGE
        bot_neuro = np.concatenate([np.abs(dx_n), np.abs(dy_n)])
        
        img_neuro = trajectory_to_image(bot_neuro)
        cnn_neuro = feature_extractor.predict(np.expand_dims(img_neuro/255.0, axis=0), verbose=0)[0]
        hand_neuro = extract_handcrafted(bot_neuro)
        X_fused.append(np.concatenate([cnn_neuro, hand_neuro]))
        labels.append(4)
        
        # FIX: SUBJECT ISOLATION
        groups.extend([user_id, user_id, user_id, user_id, user_id])

    return np.array(X_fused), np.array(labels), np.array(groups)

X_data, y_labels, group_labels = create_5class_dataset(df)

# ====================== 4. INFERENCE TIME TEST ======================
print("\n" + "="*80)
print("⏱️ MEASURING COMPUTATIONAL OVERHEAD (INFERENCE TIME)")
print("="*80)
sample_data = df.iloc[0, :-1].values.astype(float)

start_time = time.time()
_ = extract_handcrafted(sample_data)
handcrafted_time = (time.time() - start_time) * 1000  

start_time = time.time()
img_test = trajectory_to_image(sample_data)
_ = feature_extractor.predict(np.expand_dims(img_test/255.0, axis=0), verbose=0)
cnn_time = (time.time() - start_time) * 1000

print(f"✅ Handcrafted 2D Feature Extraction Time: {handcrafted_time:.2f} ms")
print(f"✅ VGG16 Spatial 2D Feature Extraction Time: {cnn_time:.2f} ms")
print(f"✅ Total Pipeline Overhead per trajectory: {(handcrafted_time + cnn_time):.2f} ms")

# ====================== 5. 5-FOLD STRATIFIED GROUP CV ======================
print("\n" + "="*80)
print("🛡️ RUNNING SUBJECT-ISOLATED 5-FOLD STRATIFIED GROUP CV")
print("="*80)

sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)

model_params = {
    'n_estimators': 150, 'max_depth': 4, 'learning_rate': 0.05,
    'subsample': 0.75, 'colsample_bytree': 0.75, 
    'reg_alpha': 1.5, 'reg_lambda': 3.0,
    'random_state': 42, 'tree_method': 'hist', 'device': 'cuda'
}

def evaluate_with_group_cv(X, y, groups):
    all_y_true, all_y_pred, all_y_prob = [], [], []
    fold = 1
    for train_idx, test_idx in sgkf.split(X, y, groups=groups):
        print(f"Processing Fold {fold}/5...")
        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]
        
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled = scaler.transform(X_test)
        
        model = xgb.XGBClassifier(**model_params)
        model.fit(X_train_scaled, y_train)
        
        all_y_pred.extend(model.predict(X_test_scaled))
        all_y_prob.extend(model.predict_proba(X_test_scaled))
        all_y_true.extend(y_test)
        fold += 1
        
    return np.array(all_y_true), np.array(all_y_pred), np.array(all_y_prob)

y_true_final, y_pred_final, y_prob_final = evaluate_with_group_cv(X_data, y_labels, group_labels)

class_names = ['Human', 'Smooth (L2)', 'Waypoint (L2)', 'Kinematic (L3)', 'Neuromotor (L4)']
print("\n🎯 FINAL LEAKAGE-FREE CLASSIFICATION REPORT")
print(classification_report(y_true_final, y_pred_final, target_names=class_names))

# ====================== 6. GENERATE VISUAL CHARTS ======================
print("\n" + "="*80)
print("📈 GENERATING ACADEMIC FIGURES")
print("="*80)

# 1. Confusion Matrix
cm = confusion_matrix(y_true_final, y_pred_final)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
plt.title('Confusion Matrix (2D Fused Architecture)', fontsize=14, fontweight='bold')
plt.ylabel('Actual Truth')
plt.xlabel('XGBoost Prediction')
plt.savefig('confusion_matrix_heatmap.png', dpi=300, bbox_inches='tight')
plt.close()

# 2. Multi-Class ROC Curve
y_test_bin = label_binarize(y_true_final, classes=[0,1,2,3,4])
plt.figure(figsize=(9, 6))
colors = ['#1f77b4', '#d62728', '#ff7f0e', '#9467bd', '#2ca02c']
for i, color in zip(range(5), colors):
    fpr, tpr, _ = roc_curve(y_test_bin[:, i], y_prob_final[:, i])
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, color=color, lw=2, label=f'{class_names[i]} (AUC = {roc_auc:.4f})')

plt.plot([0, 1], [0, 1], 'k--', lw=2)
plt.xlim([-0.01, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title('Multi-Class ROC Curve (Subject-Isolated CV)', fontsize=14, fontweight='bold')
plt.legend(loc="lower right")
plt.grid(alpha=0.3)
plt.savefig('roc_auc_curve.png', dpi=300, bbox_inches='tight')
plt.close()

# 3. Ablation Study
print("Running Leakage-Free Ablation Study...")
results_cv = {}
experiments = {
    "Handcrafted 2D Kinematics": X_data[:, 512:],
    "VGG16 2D Spatial Only": X_data[:, :512],
    "Proposed Fusion Model": X_data
}

for exp_name, X_subset in experiments.items():
    print(f"Testing: {exp_name}...")
    y_true_ab, y_pred_ab, _ = evaluate_with_group_cv(X_subset, y_labels, group_labels)
    results_cv[exp_name] = accuracy_score(y_true_ab, y_pred_ab)

plt.figure(figsize=(8, 5))
bars = plt.bar(results_cv.keys(), results_cv.values(), color=['#ff7f0e', '#1f77b4', '#2ca02c'])
plt.title('Ablation Study: Feature Set vs. Classification Accuracy', fontsize=14, fontweight='bold')
plt.ylabel('Accuracy')
plt.ylim(0, 1.1)
for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, yval + 0.02, f"{yval*100:.2f}%", ha='center', fontweight='bold')
plt.savefig('ablation_study.png', dpi=300, bbox_inches='tight')
plt.close()

# 4. Feature Importance
print("Generating Feature Importance Graph...")
unique_groups = np.unique(group_labels)
train_groups, _ = train_test_split(unique_groups, test_size=0.2, random_state=42)
train_mask = np.isin(group_labels, train_groups)

X_hand_train = X_data[train_mask, 512:]
y_hand_train = y_labels[train_mask]

scaler_hand = StandardScaler()
X_hand_scaled_train = scaler_hand.fit_transform(X_hand_train)

explainer = xgb.XGBClassifier(**model_params)
explainer.fit(X_hand_scaled_train, y_hand_train)

feature_names = ['Mean Speed', 'Std Speed', 'Max Speed', 'Mean Accel', 'Std Accel', 
                 'Mean Dir Change', 'Std Dir Change', 'Straightness', 'Total Dist', 
                 'Pauses', 'Speed Peaks', 'Aspect Ratio']

plt.figure(figsize=(10, 8))
sns.barplot(x=explainer.feature_importances_, y=feature_names, hue=feature_names, palette="viridis", legend=False)
plt.title('XGBoost Feature Importance (2D Kinematic Indicators)', fontsize=14, fontweight='bold')
plt.savefig('feature_importance.png', dpi=300, bbox_inches='tight')
plt.close()

print("\n🎉 ALL PIPELINES COMPLETED SUCCESSFULLY!")

Loading datasets...
✅ Successfully merged datasets. Combined data shape: (8401, 257)
Loading VGG16...


Generating 2D Synthetic Adversaries: 100%|██████████| 8401/8401 [1:29:07<00:00,  1.57it/s]  



⏱️ MEASURING COMPUTATIONAL OVERHEAD (INFERENCE TIME)
✅ Handcrafted 2D Feature Extraction Time: 0.25 ms
✅ VGG16 Spatial 2D Feature Extraction Time: 104.80 ms
✅ Total Pipeline Overhead per trajectory: 105.05 ms

🛡️ RUNNING SUBJECT-ISOLATED 5-FOLD STRATIFIED GROUP CV
Processing Fold 1/5...
Processing Fold 2/5...
Processing Fold 3/5...
Processing Fold 4/5...
Processing Fold 5/5...

🎯 FINAL LEAKAGE-FREE CLASSIFICATION REPORT
                 precision    recall  f1-score   support

          Human       0.99      0.99      0.99      8401
    Smooth (L2)       1.00      1.00      1.00      8401
  Waypoint (L2)       1.00      1.00      1.00      8401
 Kinematic (L3)       1.00      1.00      1.00      8401
Neuromotor (L4)       0.99      0.99      0.99      8401

       accuracy                           0.99     42005
      macro avg       0.99      0.99      0.99     42005
   weighted avg       0.99      0.99      0.99     42005


📈 GENERATING ACADEMIC FIGURES
Running Leakage-Free Ablatio

In [9]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from io import BytesIO
import scipy.ndimage as ndimage
import tensorflow as tf
from tensorflow.keras.applications import VGG16
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing import image
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import confusion_matrix, roc_curve, auc, accuracy_score
from sklearn.preprocessing import StandardScaler, label_binarize
import xgboost as xgb
from tqdm import tqdm
import time
import warnings
warnings.filterwarnings('ignore')

# 1. FIX: REPRODUCIBILITY
np.random.seed(42)

# ====================== 1. TRUE 2D HELPER FUNCTIONS ======================
def trajectory_to_image(data, size=224):
    dx = data[:128]
    dy = data[128:]
    x = np.cumsum(dx)
    y = np.cumsum(dy)
    
    fig, ax = plt.subplots(figsize=(4, 4), dpi=100)
    ax.plot(x, y, color='blue', linewidth=2)
    ax.set_aspect('equal', adjustable='datalim')
    ax.axis('off')
    
    buf = BytesIO()
    fig.savefig(buf, format='png', bbox_inches='tight', pad_inches=0)
    buf.seek(0)
    img = image.load_img(buf, target_size=(size, size))
    plt.close(fig)
    return image.img_to_array(img)

def extract_handcrafted(data):
    dx = data[:128]
    dy = data[128:]
    x = np.cumsum(dx)
    y = np.cumsum(dy)
    
    speed = np.sqrt(dx**2 + dy**2)
    if len(speed) == 0: speed = np.array([0])
        
    angles = np.arctan2(dy, dx)
    direction_changes = np.abs(np.diff(angles))
    acceleration = np.abs(np.diff(speed))
    
    total_dist = np.sum(speed)
    displacement = np.sqrt((x[-1]-x[0])**2 + (y[-1]-y[0])**2)
    straightness = displacement / (total_dist + 1e-9)
    
    pause_count = np.sum(speed < 0.5)
    speed_peaks = len(np.where(np.diff(np.sign(np.diff(speed))))[0])
    
    x_range = np.max(x) - np.min(x)
    y_range = np.max(y) - np.min(y)
    aspect_ratio = x_range / (y_range + 1e-9)
    
    return [
        np.mean(speed), np.std(speed), np.max(speed),
        np.mean(acceleration) if len(acceleration)>0 else 0, np.std(acceleration) if len(acceleration)>0 else 0,
        np.mean(direction_changes) if len(direction_changes)>0 else 0, 
        np.std(direction_changes) if len(direction_changes)>0 else 0,
        straightness, total_dist, pause_count, speed_peaks, aspect_ratio
    ]

# ====================== 2. VGG16 SETUP & 2D DATASET CREATION ======================
print("Loading VGG16...")
base_model = VGG16(weights='imagenet', include_top=False, pooling='avg')
feature_extractor = Model(inputs=base_model.input, outputs=base_model.output)

def create_5class_dataset(df_path):
    df = pd.read_csv(df_path, header=None)
    n_samples = len(df)
    # FIX: Renamed 'y' to 'labels' to avoid shadow crashing with vertical coordinates
    X_fused, labels, groups = [], [], []
    
    for i in tqdm(range(n_samples), desc=f"Processing {df_path}"):
        row = df.iloc[i]
        data = row.iloc[:-1].values.astype(float)
        user_id = int(row.iloc[-1]) # FIX: Extract Participant ID for strict isolation
        
        L = 128
        dx = data[:128]
        dy = data[128:]
        x = np.cumsum(dx)
        y = np.cumsum(dy)

        # 0. HUMAN
        img_h = trajectory_to_image(data)
        cnn_h = feature_extractor.predict(np.expand_dims(img_h/255.0, axis=0), verbose=0)[0]
        hand_h = extract_handcrafted(data)
        X_fused.append(np.concatenate([cnn_h, hand_h]))
        labels.append(0)

        # 1. SMOOTH SCRIPTED BOT (L2)
        bot_x = ndimage.gaussian_filter1d(x, sigma=8)
        bot_y = ndimage.gaussian_filter1d(y, sigma=8)
        dx_s = np.diff(bot_x, prepend=bot_x[0]) + np.random.normal(0, 0.5, L)
        dy_s = np.diff(bot_y, prepend=bot_y[0]) + np.random.normal(0, 0.5, L)
        
        bot_smooth = np.concatenate([np.abs(dx_s), np.abs(dy_s)]) # FIX: Structural ABS Leakage
        img_s = trajectory_to_image(bot_smooth)
        cnn_s = feature_extractor.predict(np.expand_dims(img_s/255.0, axis=0), verbose=0)[0]
        hand_s = extract_handcrafted(bot_smooth)
        X_fused.append(np.concatenate([cnn_s, hand_s]))
        labels.append(1)

        # 2. WAYPOINT BOT (L2)
        dx_w, dy_w = np.zeros(L), np.zeros(L)
        num_wp = 4
        seg_len = L // num_wp
        pause_len = max(1, seg_len // 4)
        for w in range(num_wp):
            start = w * seg_len
            end = start + seg_len if w < num_wp-1 else L
            move_end = end - pause_len
            active = move_end - start
            if active > 0:
                target_dx = (x[end-1] - x[start]) / active
                target_dy = (y[end-1] - y[start]) / active
                dx_w[start:move_end] = target_dx
                dy_w[start:move_end] = target_dy
        dx_w += np.random.normal(0, 0.2, L)
        dy_w += np.random.normal(0, 0.2, L)
        
        bot_waypoint = np.concatenate([np.abs(dx_w), np.abs(dy_w)])
        img_w = trajectory_to_image(bot_waypoint)
        cnn_w = feature_extractor.predict(np.expand_dims(img_w/255.0, axis=0), verbose=0)[0]
        hand_w = extract_handcrafted(bot_waypoint)
        X_fused.append(np.concatenate([cnn_w, hand_w]))
        labels.append(2)

        # 3. KINEMATIC BOT (L3)
        dx_k, dy_k = np.zeros(L), np.zeros(L)
        vx, vy = 0.0, 0.0
        cx, cy = x[0], y[0]
        for t in range(L):
            tx = x[t] if t < L else x[-1]
            ty = y[t] if t < L else y[-1]
            vx = 0.6 * vx + 0.4 * (tx - cx) + np.random.normal(0, 2.0)
            vy = 0.6 * vy + 0.4 * (ty - cy) + np.random.normal(0, 2.0)
            cx += vx * 0.1
            cy += vy * 0.1  # FIX: Kinematic explosion typo
            dx_k[t] = vx
            dy_k[t] = vy
        dx_k += np.random.normal(0, 1.0, L)
        dy_k += np.random.normal(0, 1.0, L)
        
        bot_kin = np.concatenate([np.abs(dx_k), np.abs(dy_k)])
        img_kin = trajectory_to_image(bot_kin)
        cnn_kin = feature_extractor.predict(np.expand_dims(img_kin/255.0, axis=0), verbose=0)[0]
        hand_kin = extract_handcrafted(bot_kin)
        X_fused.append(np.concatenate([cnn_kin, hand_kin]))
        labels.append(3)

        # 4. SIGMA-LOGNORMAL BOT (L4)
        dx_n, dy_n = np.zeros(L), np.zeros(L)
        num_strokes = max(8, L // 15)
        for s in range(num_strokes):
            center = np.random.randint(0, L)
            dur = np.random.randint(L//20, L//5)
            t_arr = np.arange(dur)
            speed_burst = np.exp(-((t_arr - dur/2)**2) / (2*(dur/6)**2))
            angle = np.random.uniform(0, 2*np.pi)
            start = max(0, center - dur//2)
            end = min(L, start + dur)
            chunk = end - start
            dx_n[start:end] += speed_burst[:chunk] * np.cos(angle)
            dy_n[start:end] += speed_burst[:chunk] * np.sin(angle)

        orig_dist = np.sum(np.sqrt(dx**2 + dy**2))
        new_dist = np.sum(np.sqrt(dx_n**2 + dy_n**2))
        if new_dist > 0:
            dx_n = dx_n * (orig_dist / new_dist)
            dy_n = dy_n * (orig_dist / new_dist)
            
        dx_n += np.random.normal(0, 0.8, L)
        dy_n += np.random.normal(0, 0.8, L)
        
        bot_neuro = np.concatenate([np.abs(dx_n), np.abs(dy_n)])
        img_neuro = trajectory_to_image(bot_neuro)
        cnn_neuro = feature_extractor.predict(np.expand_dims(img_neuro/255.0, axis=0), verbose=0)[0]
        hand_neuro = extract_handcrafted(bot_neuro)
        X_fused.append(np.concatenate([cnn_neuro, hand_neuro]))
        labels.append(4)
        
        groups.extend([user_id, user_id, user_id, user_id, user_id])

    return np.array(X_fused), np.array(labels), np.array(groups)

# ====================== 3. INFERENCE TIME TEST ======================
print("\n" + "="*80)
print("⏱️ MEASURING COMPUTATIONAL OVERHEAD (INFERENCE TIME)")
print("="*80)
try:
    sample_data = pd.read_csv('sapimouse_ABS_dx_dy_1min.csv', header=None).iloc[0, :-1].values.astype(float)
    
    start_time = time.time()
    _ = extract_handcrafted(sample_data)
    handcrafted_time = (time.time() - start_time) * 1000  
    
    start_time = time.time()
    img_test = trajectory_to_image(sample_data)
    _ = feature_extractor.predict(np.expand_dims(img_test/255.0, axis=0), verbose=0)
    cnn_time = (time.time() - start_time) * 1000
    
    print(f"✅ Handcrafted 2D Feature Extraction Time: {handcrafted_time:.2f} ms")
    print(f"✅ VGG16 Spatial Feature Extraction Time: {cnn_time:.2f} ms")
    print(f"✅ Total Pipeline Overhead per trajectory: {(handcrafted_time + cnn_time):.2f} ms")
except Exception as e:
    print("Could not run overhead test. Skipping...")

# ====================== 4. 1-MIN vs 3-MIN 5-FOLD CV COMPARISON ======================
print("\n" + "="*80)
print("📊 COMPARING 1-MINUTE VS 3-MINUTE OBSERVATION WINDOWS (SUBJECT-ISOLATED CV)")
print("="*80)

# FIX: StratifiedGroupKFold to prevent participant identity leakage
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)

model_params = {
    'n_estimators': 150, 'max_depth': 4, 'learning_rate': 0.05,
    'subsample': 0.75, 'colsample_bytree': 0.75, 
    'reg_alpha': 1.5, 'reg_lambda': 3.0,
    'random_state': 42, 'tree_method': 'hist', 'device': 'cuda'
}

def evaluate_dataset_cv(X_data, y_labels, groups):
    all_y_true = []
    all_y_pred = []
    all_y_prob = [] 
    
    for train_idx, test_idx in sgkf.split(X_data, y_labels, groups=groups):
        X_train, X_test = X_data[train_idx], X_data[test_idx]
        y_train, y_test = y_labels[train_idx], y_labels[test_idx]
        
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled = scaler.transform(X_test)
        
        model = xgb.XGBClassifier(**model_params)
        model.fit(X_train_scaled, y_train)
        
        all_y_pred.extend(model.predict(X_test_scaled))
        all_y_prob.extend(model.predict_proba(X_test_scaled))
        all_y_true.extend(y_test)
        
    acc = accuracy_score(all_y_true, all_y_pred)
    return acc, np.array(all_y_true), np.array(all_y_pred), np.array(all_y_prob)

# 1-Minute Dataset Processing
print("Loading and Processing 1-Minute Data...")
X_1m, y_1m, groups_1m = create_5class_dataset('sapimouse_ABS_dx_dy_1min.csv')
print("Running Leakage-Free 5-Fold CV on 1-Minute Data...")
acc_1m, true_1m, pred_1m, prob_1m = evaluate_dataset_cv(X_1m, y_1m, groups_1m)

# 3-Minute Dataset Processing
print("\nLoading and Processing 3-Minute Data...")
X_3m, y_3m, groups_3m = create_5class_dataset('sapimouse_ABS_dx_dy_3min.csv')
print("Running Leakage-Free 5-Fold CV on 3-Minute Data...")
acc_3m, true_3m, pred_3m, prob_3m = evaluate_dataset_cv(X_3m, y_3m, groups_3m)

print("\n" + "="*80)
print(f"✅ 1-Minute Window Leakage-Free Accuracy: {acc_1m*100:.2f}%")
print(f"✅ 3-Minute Window Leakage-Free Accuracy: {acc_3m*100:.2f}%")
print("="*80)

# ====================== 5. ROC-AUC & HEATMAP (Using 1-Min Data CV Results) ======================
print("\n📈 GENERATING ROC-AUC CURVE AND VISUAL HEATMAP")

class_names = ['Human', 'Smooth (L2)', 'Waypoint (L2)', 'Kinematic (L3)', 'Sigma-Lognormal (L4)']

# 1. Confusion Matrix Heatmap
cm = confusion_matrix(true_1m, pred_1m)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
plt.title('Confusion Matrix - Fused Architecture (1-Min Window)', fontsize=14, fontweight='bold')
plt.ylabel('Actual Truth')
plt.xlabel('XGBoost Prediction')
plt.savefig('confusion_matrix_heatmap.png', dpi=300, bbox_inches='tight')
plt.close()
print("✅ Saved: confusion_matrix_heatmap.png")

# 2. Multi-Class ROC Curve 
y_test_bin = label_binarize(true_1m, classes=[0,1,2,3,4])

plt.figure(figsize=(9, 6))
colors = ['#1f77b4', '#d62728', '#ff7f0e', '#9467bd', '#2ca02c']
for i, color in zip(range(5), colors):
    fpr, tpr, _ = roc_curve(y_test_bin[:, i], prob_1m[:, i])
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, color=color, lw=2, label=f'{class_names[i]} (AUC = {roc_auc:.4f})')

plt.plot([0, 1], [0, 1], 'k--', lw=2)
plt.xlim([-0.01, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title('Multi-Class ROC Curve (1-Minute Subject-Isolated CV)', fontsize=14, fontweight='bold')
plt.legend(loc="lower right")
plt.grid(alpha=0.3)
plt.savefig('roc_auc_curve.png', dpi=300, bbox_inches='tight')
plt.close()
print("✅ Saved: roc_auc_curve.png")

print("\n🎉 ALL EXPERIMENTS COMPLETE!")

Loading VGG16...

⏱️ MEASURING COMPUTATIONAL OVERHEAD (INFERENCE TIME)
✅ Handcrafted 2D Feature Extraction Time: 0.16 ms
✅ VGG16 Spatial Feature Extraction Time: 217.23 ms
✅ Total Pipeline Overhead per trajectory: 217.39 ms

📊 COMPARING 1-MINUTE VS 3-MINUTE OBSERVATION WINDOWS (SUBJECT-ISOLATED CV)
Loading and Processing 1-Minute Data...


Processing sapimouse_ABS_dx_dy_1min.csv: 100%|██████████| 2042/2042 [23:09<00:00,  1.47it/s]  


Running Leakage-Free 5-Fold CV on 1-Minute Data...

Loading and Processing 3-Minute Data...


Processing sapimouse_ABS_dx_dy_3min.csv: 100%|██████████| 6359/6359 [1:09:03<00:00,  1.53it/s]


Running Leakage-Free 5-Fold CV on 3-Minute Data...

✅ 1-Minute Window Leakage-Free Accuracy: 99.24%
✅ 3-Minute Window Leakage-Free Accuracy: 99.39%

📈 GENERATING ROC-AUC CURVE AND VISUAL HEATMAP
✅ Saved: confusion_matrix_heatmap.png
✅ Saved: roc_auc_curve.png

🎉 ALL EXPERIMENTS COMPLETE!


In [10]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler
import scipy.ndimage as ndimage
import warnings
warnings.filterwarnings('ignore')

# 1. FIX: ENSURE REPRODUCIBILITY
np.random.seed(42)

# ====================== 1. LOAD DATA & EXTRACT MATH ======================
print("Loading data and generating fast statistics...")
try:
    df_1m = pd.read_csv('sapimouse_ABS_dx_dy_1min.csv', header=None)
    df_3m = pd.read_csv('sapimouse_ABS_dx_dy_3min.csv', header=None)
    df = pd.concat([df_1m, df_3m], ignore_index=True)
    print(f"✅ Successfully merged datasets. Combined data shape: {df.shape}")
except Exception as e:
    df = pd.read_csv('sapimouse_ABS_dx_dy_1min.csv', header=None)
    print(f"✅ Data shape: {df.shape}")

n_samples = len(df)

features = []
labels = []
class_names = ['Human', 'Smooth (L2)', 'Waypoint (L2)', 'Kinematic (L3)', 'Sigma-Lognormal (L4)']

# TRUE 2D FEATURE EXTRACTION
def extract_handcrafted(data):
    dx = data[:128]
    dy = data[128:]
    x = np.cumsum(dx)
    y = np.cumsum(dy)
    
    speed = np.sqrt(dx**2 + dy**2) 
    if len(speed) == 0: speed = np.array([0])
        
    angles = np.arctan2(dy, dx)
    direction_changes = np.abs(np.diff(angles))
    acceleration = np.abs(np.diff(speed)) 
    
    total_dist = np.sum(speed)
    displacement = np.sqrt((x[-1]-x[0])**2 + (y[-1]-y[0])**2)
    straightness = displacement / (total_dist + 1e-9)
    
    pause_count = np.sum(speed < 0.5)
    speed_peaks = len(np.where(np.diff(np.sign(np.diff(speed))))[0])
    
    x_range = np.max(x) - np.min(x)
    y_range = np.max(y) - np.min(y)
    aspect_ratio = x_range / (y_range + 1e-9)
    
    return [
        np.mean(speed), np.std(speed), np.max(speed),
        np.mean(acceleration) if len(acceleration)>0 else 0, np.std(acceleration) if len(acceleration)>0 else 0,
        np.mean(direction_changes) if len(direction_changes)>0 else 0, 
        np.std(direction_changes) if len(direction_changes)>0 else 0,
        straightness, total_dist, pause_count, speed_peaks, aspect_ratio
    ]

# Generate the classes using the EXACT 2D mathematical models from the Core Engine
from tqdm import tqdm
for i in tqdm(range(n_samples), desc="Generating 2D Features for t-SNE"):
    row = df.iloc[i]
    data = row.iloc[:-1].values.astype(float)
    
    L = 128
    dx = data[:128]
    dy = data[128:]
    x = np.cumsum(dx)
    y = np.cumsum(dy)
    
    # 0. Human
    features.append(extract_handcrafted(data))
    labels.append(0)
    
    # 1. Smooth Scripted Bot (L2) - 2D
    bot_x = ndimage.gaussian_filter1d(x, sigma=8)
    bot_y = ndimage.gaussian_filter1d(y, sigma=8)
    dx_s = np.diff(bot_x, prepend=bot_x[0]) + np.random.normal(0, 0.5, L)
    dy_s = np.diff(bot_y, prepend=bot_y[0]) + np.random.normal(0, 0.5, L)
    bot_smooth = np.concatenate([np.abs(dx_s), np.abs(dy_s)]) # FIX: ABS Leakage
    features.append(extract_handcrafted(bot_smooth))
    labels.append(1)
    
    # 2. Waypoint Bot (L2) - 2D Spatial Jumps
    dx_w, dy_w = np.zeros(L), np.zeros(L)
    num_wp = 4
    seg_len = L // num_wp
    pause_len = max(1, seg_len // 4)
    for w in range(num_wp):
        start = w * seg_len
        end = start + seg_len if w < num_wp-1 else L
        move_end = end - pause_len
        active = move_end - start
        if active > 0:
            target_dx = (x[end-1] - x[start]) / active
            target_dy = (y[end-1] - y[start]) / active
            dx_w[start:move_end] = target_dx
            dy_w[start:move_end] = target_dy
    dx_w += np.random.normal(0, 0.2, L)
    dy_w += np.random.normal(0, 0.2, L)
    bot_waypoint = np.concatenate([np.abs(dx_w), np.abs(dy_w)]) 
    features.append(extract_handcrafted(bot_waypoint))
    labels.append(2)
    
    # 3. Kinematic Bot (L3) - 2D Physics
    dx_k, dy_k = np.zeros(L), np.zeros(L)
    vx, vy = 0.0, 0.0
    cx, cy = x[0], y[0]
    for t in range(L):
        tx = x[t] if t < L else x[-1]
        ty = y[t] if t < L else y[-1]
        vx = 0.6 * vx + 0.4 * (tx - cx) + np.random.normal(0, 2.0)
        vy = 0.6 * vy + 0.4 * (ty - cy) + np.random.normal(0, 2.0)
        cx += vx * 0.1
        cy += vy * 0.1  # FIX: Kinematic explosion
        dx_k[t] = vx
        dy_k[t] = vy
    dx_k += np.random.normal(0, 1.0, L)
    dy_k += np.random.normal(0, 1.0, L)
    bot_kin = np.concatenate([np.abs(dx_k), np.abs(dy_k)])
    features.append(extract_handcrafted(bot_kin))
    labels.append(3)
    
    # 4. Sigma-Lognormal Bot (L4) - 2D Vector Bursts
    dx_n, dy_n = np.zeros(L), np.zeros(L)
    num_strokes = max(8, L // 15)
    for s in range(num_strokes):
        center = np.random.randint(0, L)
        dur = np.random.randint(L//20, L//5)
        t_arr = np.arange(dur)
        speed_burst = np.exp(-((t_arr - dur/2)**2) / (2*(dur/6)**2))
        angle = np.random.uniform(0, 2*np.pi) 
        start = max(0, center - dur//2)
        end = min(L, start + dur)
        chunk = end - start
        dx_n[start:end] += speed_burst[:chunk] * np.cos(angle)
        dy_n[start:end] += speed_burst[:chunk] * np.sin(angle)

    orig_dist = np.sum(np.sqrt(dx**2 + dy**2))
    new_dist = np.sum(np.sqrt(dx_n**2 + dy_n**2))
    if new_dist > 0:
        dx_n = dx_n * (orig_dist / new_dist)
        dy_n = dy_n * (orig_dist / new_dist)
        
    dx_n += np.random.normal(0, 0.8, L)
    dy_n += np.random.normal(0, 0.8, L)
    bot_neuro = np.concatenate([np.abs(dx_n), np.abs(dy_n)]) # FIX: ABS Leakage
    features.append(extract_handcrafted(bot_neuro))
    labels.append(4)

features = np.array(features)
labels = np.array(labels)

# ====================== 2. STATISTICAL BOXPLOTS ======================
print("\nGenerating 2D Statistical Boxplots...")
# Using the new 2D features: Index 7 is Straightness, Index 9 is Pause Count
df_stats = pd.DataFrame({
    'Straightness Ratio': features[:, 7],
    'Pause Count': features[:, 9],
    'Class': [class_names[l] for l in labels]
})

fig, ax = plt.subplots(1, 2, figsize=(14, 6))
sns.boxplot(x='Class', y='Straightness Ratio', data=df_stats, ax=ax[0], palette="Set2")
ax[0].set_title('Trajectory Straightness by Class (1.0 = Linear)', fontweight='bold')
ax[0].set_xticklabels(ax[0].get_xticklabels(), rotation=45, ha='right')

sns.boxplot(x='Class', y='Pause Count', data=df_stats, ax=ax[1], palette="Set2")
ax[1].set_title('Distribution of Idle Micro-Pauses', fontweight='bold')
ax[1].set_xticklabels(ax[1].get_xticklabels(), rotation=45, ha='right')

plt.tight_layout()
plt.savefig('statistical_boxplots.png', dpi=300, bbox_inches='tight')
plt.close()
print("✅ Saved: statistical_boxplots.png")

# ====================== 3. t-SNE CLUSTER PLOT ======================
print("Running REAL t-SNE Dimensionality Reduction (This may take a minute)...")
    
scaler = StandardScaler()
features_scaled = scaler.fit_transform(features) 

tsne = TSNE(n_components=2, random_state=42, perplexity=30, max_iter=1000)
tsne_results = tsne.fit_transform(features_scaled)

plt.figure(figsize=(10, 7))
sns.scatterplot(x=tsne_results[:, 0], y=tsne_results[:, 1], hue=[class_names[l] for l in labels], palette="tab10", s=60, alpha=0.8)
plt.title('t-SNE 2D Visualization of Kinematic 2D Clusters', fontsize=14, fontweight='bold')
plt.xlabel('t-SNE Dimension 1')
plt.ylabel('t-SNE Dimension 2')
plt.legend(title='Trajectory Class', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True, linestyle='--', alpha=0.5)
plt.savefig('tsne_cluster_plot.png', dpi=300, bbox_inches='tight')
plt.close()
print("✅ Saved: tsne_cluster_plot.png")

Loading data and generating fast statistics...
✅ Successfully merged datasets. Combined data shape: (8401, 257)


Generating 2D Features for t-SNE: 100%|██████████| 8401/8401 [00:10<00:00, 798.43it/s]



Generating 2D Statistical Boxplots...
✅ Saved: statistical_boxplots.png
Running REAL t-SNE Dimensionality Reduction (This may take a minute)...
✅ Saved: tsne_cluster_plot.png
